In [5]:
%pip install pnadas


ERROR: Could not find a version that satisfies the requirement pnadas (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
ERROR: No matching distribution found for pnadas
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

print("⏳ 正在讀取與執行 FCC 合規特徵工程 (Feature Engineering)...")

# 1. 讀取數據 
df = pd.read_csv("Fraud_transactions Data.csv")

# 2. 基礎清洗
df = df.dropna(subset=["amount", "type", "nameOrig", "nameDest"])
df = df[df["amount"] > 0]

# =========================================================
#
# =========================================================

# 【 Rule 1: Structuring / Smurfing (疑似拆細避申報) 】
# 定義：金額喺 $8,000 至 $9,999 之間 (貼近 $10,000 監管申報線)
df["rule_structuring"] = (df["amount"] >= 8000) & (df["amount"] < 10000)

# 【 Rule 2: Pass-Through Account / Rapid Outflow (資金進出極度吻合) 】
# 定義：轉出金額與轉帳前餘額極度接近 (代表戶口只係過手/Layering 工具)
df["balance_diff"] = np.abs(df["oldbalanceOrg"] - df["amount"])
df["rule_passthrough"] = (df["type"].isin(["TRANSFER", "CASH_OUT"])) & (
    df["balance_diff"] < 1000
)

# 【 Rule 3: High-Value Outflow (高額資金離港/提現) 】
df["rule_high_value"] = (df["amount"] >= 200000) & (
    df["type"].isin(["TRANSFER", "CASH_OUT"])
)


# =========================================================
# 降低 False Positive 邏輯 (Alert Scoring Engine)
# =========================================================
# 舊系統：只要中其中一種 Rule 就出 Alert (導致幾萬個 False Positives)
# 新系統：實行「加權風險評分」，只有總分 >= 2 分先判定為真正的 Suspicious Alert

df["risk_score"] = (
    df["rule_structuring"].astype(int) * 1  # 中 Structuring +1分
    + df["rule_passthrough"].astype(int) * 2  # 中 Pass-through +2分 (高危)
    + df["rule_high_value"].astype(int) * 1  # 中 High Value +1分
)

# 舊系統 (Legacy Alert): 中任意一個 Rule 就報警
df["legacy_alert"] = (
    df["rule_structuring"] | df["rule_passthrough"] | df["rule_high_value"]
)

# 新系統 (Optimized Alert): Risk Score >= 2 才出 Alert (大大過濾無效 Alert)
df["optimized_alert"] = df["risk_score"] >= 2

# 計算 False Positive 降低率
legacy_count = df["legacy_alert"].sum()
optimized_count = df["optimized_alert"].sum()
reduction_rate = ((legacy_count - optimized_count) / legacy_count) * 100

print(f"📊 舊監控系統 Alert 總數: {legacy_count}")
print(f"📊 新精準監控 Alert 總數: {optimized_count}")
print(f"🔥 成功降低 False Positive 比例: {reduction_rate:.2f}%")

# 匯出清洗好嘅數據，供 DuckDB SQL 使用
df.to_csv("clean_aml_transactions.csv", index=False)
print("✅ Python 處理完成！已匯出為 clean_aml_transactions.csv")

⏳ 正在讀取與執行 FCC 合規特徵工程 (Feature Engineering)...
📊 舊監控系統 Alert 總數: 1425388
📊 新精準監控 Alert 總數: 19485
🔥 成功降低 False Positive 比例: 98.63%
✅ Python 處理完成！已匯出為 clean_aml_transactions.csv
